# Analysis: feature-vector proof

Loads every artefact the proof produced into dataframes. Nothing is analysed
here; the analysis is yours.

Source: `proof_feature.ipynb`. The recorded study is 200 trials, 36 of them completed, best at 100% lap completion and 0% off-track.

In [ ]:
import glob
import json
import os
import sqlite3

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

In [ ]:
ROOT = "runs/proof"
STUDY = "proof_feature"

In [ ]:
def load_trials(root: str, study: str) -> pd.DataFrame:
    """Every Optuna trial with its params, outcome and recorded reason.

    Pruned and failed trials are included: the reason a trial stopped is a
    result, not an absence of one.

    Args:
        root: Runs root holding ``study.db``.
        study: Study name inside that database.

    Returns:
        One row per trial.
    """
    con = sqlite3.connect(f"file:{root}/study.db?mode=ro", uri=True)
    sid = con.execute("select study_id from studies where study_name=?",
                      (study,)).fetchone()[0]
    rows = []
    for tid, number, state in con.execute(
            "select trial_id,number,state from trials where study_id=? "
            "order by number", (sid,)):
        attrs = {k: json.loads(v) for k, v in con.execute(
            "select key,value_json from trial_user_attributes where trial_id=?",
            (tid,))}
        dists = {k: json.loads(d) for k, d in con.execute(
            "select param_name,distribution_json from trial_params "
            "where trial_id=?", (tid,))}
        params = {}
        for name, value in con.execute(
                "select param_name,param_value from trial_params where trial_id=?",
                (tid,)):
            choices = dists[name]["attributes"].get("choices")
            params[name] = (choices[int(value)] if choices
                            else (int(value) if dists[name]["name"] ==
                                  "IntDistribution" else value))
        value = con.execute("select value from trial_values where trial_id=?",
                            (tid,)).fetchone()
        rows.append({"trial": number, "state": state,
                     "outcome": attrs.get("outcome", "unrecorded"),
                     "reason": attrs.get("reason", ""),
                     "score": value[0] if value else None,
                     "completion": attrs.get("completion"),
                     "progress_m": attrs.get("progress_m"), **params})
    con.close()
    return pd.DataFrame(rows)


def load_curves(root: str, study: str) -> pd.DataFrame:
    """The progress trajectory Hyperband saw, for every trial including pruned.

    Args:
        root: Runs root holding ``study.db``.
        study: Study name inside that database.

    Returns:
        Long form: one row per (trial, eval step).
    """
    con = sqlite3.connect(f"file:{root}/study.db?mode=ro", uri=True)
    rows = con.execute(
        "select t.number, v.step, v.intermediate_value "
        "from trial_intermediate_values v "
        "join trials t on t.trial_id = v.trial_id "
        "join studies s on s.study_id = t.study_id "
        "where s.study_name=? order by t.number, v.step", (study,)).fetchall()
    con.close()
    return pd.DataFrame(rows, columns=["trial", "step", "progress_m"])


def load_evals(root: str) -> pd.DataFrame:
    """Every periodic evaluation from every finished run under ``root``.

    Args:
        root: Runs root scanned recursively for ``eval_record.json``.

    Returns:
        Long form: one row per (run, eval), with the run's spec context.
    """
    rows = []
    for path in sorted(glob.glob(f"{root}/**/eval_record.json", recursive=True)):
        record = json.load(open(path))
        env, ppo = record["spec"]["env"], record["spec"]["algorithm"]["ppo"]
        context = {"run": os.path.basename(os.path.dirname(path)),
                   "variant": record.get("variant"),
                   "reward": env.get("reward"),
                   "num_envs": env.get("num_envs"),
                   "resolution": tuple(env.get("resolution") or ()),
                   "frame_stack": env.get("frame_stack"),
                   "max_laps": env.get("max_laps"),
                   "lr": ppo.get("lr"), "gamma": ppo.get("gamma"),
                   "minibatches": ppo.get("minibatches"),
                   "epochs": ppo.get("epochs")}
        for step in record.get("eval_history") or []:
            rows.append({**context, **step, "final": False})
        rows.append({**context, **record["metrics"],
                     "frames": record["train"].get("total_env_steps"),
                     "final": True})
    return pd.DataFrame(rows)


def load_finals(root: str) -> pd.DataFrame:
    """One row per finished run: final metrics, spec context and throughput.

    Args:
        root: Runs root scanned recursively for ``eval_record.json``.

    Returns:
        One row per run directory.
    """
    rows = []
    for path in sorted(glob.glob(f"{root}/**/eval_record.json", recursive=True)):
        record = json.load(open(path))
        env, ppo = record["spec"]["env"], record["spec"]["algorithm"]["ppo"]
        rows.append({"run": os.path.basename(os.path.dirname(path)),
                     "run_dir": os.path.dirname(path),
                     "variant": record.get("variant"),
                     "spec_id": record.get("spec_id"),
                     "reward": env.get("reward"),
                     "num_envs": env.get("num_envs"),
                     "resolution": tuple(env.get("resolution") or ()),
                     "frame_stack": env.get("frame_stack"),
                     "max_laps": env.get("max_laps"),
                     "episode_length_s": env.get("episode_length_s"),
                     **{k: ppo.get(k) for k in
                        ("lr", "gamma", "minibatches", "epochs", "clip",
                         "entropy_coef")},
                     **record["metrics"], **record["train"]})
    return pd.DataFrame(rows)


def telemetry_files(run_dir: str) -> list:
    """Per-step telemetry parquets for one run, oldest eval first.

    Args:
        run_dir: A run directory from ``finals.run_dir``.

    Returns:
        Sorted parquet paths.
    """
    return sorted(glob.glob(os.path.join(run_dir, "telemetry", "*.parquet")))


def load_telemetry(run_dir: str, which: int = -1, columns=None) -> pd.DataFrame:
    """One eval's per-step telemetry. Loaded on demand: these are ~130 MB each.

    Columns: step, env, episode, track, x, y, yaw, steer, throttle, speed,
    reward, progress_delta, progress_m, lateral, half_width, off_track, done,
    track_len, dt.

    Args:
        run_dir: A run directory from ``finals.run_dir``.
        which: Index into ``telemetry_files``; -1 is the last eval.
        columns: Subset of columns to read, or None for all.

    Returns:
        One row per (env, step).
    """
    return pd.read_parquet(telemetry_files(run_dir)[which], columns=columns)

## The frames

| name | grain | what it answers |
| --- | --- | --- |
| `trials` | one row per Optuna trial | which configs were tried, what each scored, and why the ones that stopped stopped |
| `curves` | (trial, eval step) | the progress trajectory Hyperband ranked on, **including pruned trials** |
| `evals` | (run, eval) | every periodic evaluation, plus a `final=True` row per run |
| `finals` | one row per finished run | end-state metrics, spec context and throughput |

Per-step telemetry is not loaded: it is ~130 MB per eval and tens of GB in
total. Use `load_telemetry(run_dir)` on a row of `finals` when you want it.

In [ ]:
trials = load_trials(ROOT, STUDY)
curves = load_curves(ROOT, STUDY)
evals = load_evals(ROOT)
finals = load_finals(ROOT)

for name, frame in (("trials", trials), ("curves", curves),
                    ("evals", evals), ("finals", finals)):
    print(f"{name:8} {frame.shape[0]:6,} rows x {frame.shape[1]:3} cols")
print()
print("outcomes:", trials["outcome"].value_counts().to_dict())
print("telemetry:", sum(len(telemetry_files(d)) for d in finals["run_dir"]),
      "parquet files across", len(finals), "runs")

In [ ]:
trials.head()

In [ ]:
finals.head()

## Your analysis